## Setup: Mount Google Drive and Define Paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/DR_Project'
MODEL_PATH = f'{PROJECT_PATH}/models'

import os
os.makedirs(MODEL_PATH, exist_ok=True)
print("Drive mounted. Models will save to:", MODEL_PATH)

Mounted at /content/drive
Drive mounted. Models will save to: /content/drive/MyDrive/DR_Project/models


## Setup: Install Libraries

In [ ]:
!pip install xgboost shap ucimlrepo kaggle -q
import warnings
warnings.filterwarnings('ignore')
print("Libraries installed.")

Libraries installed.


## Data Loading: Diabetes 130-US Hospitals (UCI Repo)

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

diabetes_130 = fetch_ucirepo(id=296)
df_130 = diabetes_130.data.features
df_130['readmitted'] = diabetes_130.data.targets

print("Diabetes 130-US Hospitals downloaded:", df_130.shape)
df_130.head()

Diabetes 130-US Hospitals downloaded: (101766, 48)


,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,Caucasian,Female,[0-10),NaN,6,25,1,1,NaN,Pediatrics-Endocrinology,...,No,No,No,No,No,No,No,No,No,NO
1,Caucasian,Female,[10-20),NaN,1,1,7,3,NaN,NaN,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,NaN,NaN,...,No,No,No,No,No,No,No,No,Yes,NO
3,Caucasian,Male,[30-40),NaN,1,1,7,2,NaN,NaN,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,Caucasian,Male,[40-50),NaN,1,1,7,1,NaN,NaN,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## Data Loading: DiaBD Bangladesh (Mendeley Download)

In [ ]:
import urllib.request
import zipfile
import os

# Version 2 - confirmed real link from the page's "Download All" button
url = "https://data.mendeley.com/public-api/zip/m8cgwxs9s6/download/2"

# Add browser-like headers so the server doesn't block the request
req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'})

with urllib.request.urlopen(req) as response, open("diabd.zip", "wb") as out_file:
    out_file.write(response.read())

print("Downloaded successfully")

# Unzip it
with zipfile.ZipFile("diabd.zip", 'r') as zip_ref:
    zip_ref.extractall("diabd_extracted")

# List what's inside to find the exact CSV filename
for f in os.listdir("diabd_extracted"):
    print(f)

Downloaded successfully
DiaBD A Diabetes Dataset for Enhanced Risk Analysi


## Data Loading: Find CSV in Extracted DiaBD Folder

In [ ]:
import os

# Walk through everything inside the extracted folder, including subfolders
for root, dirs, files in os.walk("diabd_extracted"):
    for f in files:
        print(os.path.join(root, f))

diabd_extracted/DiaBD A Diabetes Dataset for Enhanced Risk Analysi/DiaBD_A Diabetes Dataset for Enhanced Risk Analysis and Research in Bangladesh.csv


## Data Loading: Read DiaBD Dataset

In [ ]:
import pandas as pd

# Find the CSV file anywhere inside the folder (case-insensitive, any depth)
csv_path = None
for root, dirs, files in os.walk("diabd_extracted"):
    for f in files:
        if f.lower().endswith('.csv'):
            csv_path = os.path.join(root, f)
            break

print("Found file at:", csv_path)

df_diabd = pd.read_csv(csv_path)
print("DiaBD Bangladesh downloaded:", df_diabd.shape)
df_diabd.head()

Found file at: diabd_extracted/DiaBD A Diabetes Dataset for Enhanced Risk Analysi/DiaBD_A Diabetes Dataset for Enhanced Risk Analysis and Research in Bangladesh.csv
DiaBD Bangladesh downloaded: (5288, 15)


,age,gender,pulse_rate,systolic_bp,diastolic_bp,glucose,height,weight,bmi,family_diabetes,hypertensive,family_hypertension,cardiovascular_disease,stroke,diabetic
0,42,Female,66,110,73,5.88,1.65,70.2,25.75,0,0,0,0,0,No
1,35,Female,60,125,68,5.71,1.47,42.5,19.58,0,0,0,0,0,No
2,62,Female,57,127,74,6.85,1.52,47.0,20.24,0,0,0,0,0,No
3,73,Male,55,193,112,6.28,1.63,57.4,21.72,0,0,0,0,0,No
4,68,Female,71,150,81,5.71,1.42,36.0,17.79,0,0,0,0,0,No


## Setup: Configure Kaggle API Token

In [ ]:
import os

os.environ['KAGGLE_API_TOKEN'] = 'KGAT_45f59280b0d3b10249e269b0dc642c78'

!pip install kaggle -q
print("Kaggle token set.")

Kaggle token set.


## Kaggle: List Competitions (Verification)

In [ ]:
!kaggle competitions list

ref                                                                           deadline             category         reward  teamCount  userHasEntered  
----------------------------------------------------------------------------  -------------------  --------  -------------  ---------  --------------  
https://www.kaggle.com/competitions/passenger-screening-algorithm-challenge   2017-12-15 23:59:00  Featured  1,500,000 Usd        518           False  
https://www.kaggle.com/competitions/zillow-prize-1                            2018-01-10 15:59:00  Featured  1,200,000 Usd       3770           False  
https://www.kaggle.com/competitions/data-science-bowl-2017                    2017-04-12 23:59:00  Featured  1,000,000 Usd       1972           False  
https://www.kaggle.com/competitions/vesuvius-challenge-ink-detection          2023-06-14 23:59:00  Featured  1,000,000 Usd       1249           False  
https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-3                  2026-11-02

## Data Loading: PIMA Indians Diabetes Database (Kaggle)

In [ ]:
!kaggle datasets download -d uciml/pima-indians-diabetes-database -p /content/pima --unzip

df_pima = pd.read_csv('/content/pima/diabetes.csv')
print("PIMA downloaded:", df_pima.shape)
df_pima.head()

Dataset URL: https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database
License(s): CC0-1.0
100% 8.91k/8.91k [00:00<00:00, 31.0MB/s]

PIMA downloaded: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## Data Loading: APTOS 2019 Blindness Detection (Kaggle)

In [ ]:
!kaggle competitions download -c aptos2019-blindness-detection -p /content/aptos

100% 9.51G/9.51G [01:46<00:00, 95.5MB/s]



## Data Loading: Unzip and Read APTOS Dataset

In [ ]:
!unzip -q /content/aptos/aptos2019-blindness-detection.zip -d /content/aptos

df_aptos = pd.read_csv('/content/aptos/train.csv')
print("APTOS 2019 downloaded:", df_aptos.shape)
df_aptos.head()

APTOS 2019 downloaded: (3662, 2)


,id_code,diagnosis
0,000c1434d8d7,2
1,001639a390f0,4
2,0024cdab0c1e,1
3,002c21358ce6,0
4,005b95c28852,0


## Data Persistence: Save All Datasets to Google Drive

In [ ]:
import os
import shutil

DRIVE_DATA = '/content/drive/MyDrive/DR_Project/data/clinical'
os.makedirs(DRIVE_DATA, exist_ok=True)
os.makedirs('/content/drive/MyDrive/DR_Project/data/aptos', exist_ok=True)

# Save clinical datasets as clean CSVs
df_130.to_csv(f'{DRIVE_DATA}/diabetes_130_hospitals.csv', index=False)
df_diabd.to_csv(f'{DRIVE_DATA}/diabd_bangladesh.csv', index=False)
df_pima.to_csv(f'{DRIVE_DATA}/pima_indians.csv', index=False)

print("Clinical datasets saved to Drive.")

# Copy APTOS images and labels (this will take a few minutes — it's ~10GB)
shutil.copytree('/content/aptos/train_images', '/content/drive/MyDrive/DR_Project/data/aptos/train_images', dirs_exist_ok=True)
df_aptos.to_csv('/content/drive/MyDrive/DR_Project/data/aptos/train.csv', index=False)

print("APTOS 2019 saved to Drive.")
print("All datasets backed up successfully.")

Clinical datasets saved to Drive.
APTOS 2019 saved to Drive.
All datasets backed up successfully.
